# MOOC 00 — Carte du projet

But : comprendre où sont les pièces importantes du dépôt, comment elles s'enchaînent, et où aller quand on veut modifier une expérience.

À la fin du notebook, tu dois pouvoir répondre à trois questions :

1. Quelle est la boucle expérimentale complète ?
2. Où sont les configs, les modèles, les métriques et les scripts post-hoc ?
3. Quels fichiers lire avant de toucher à une expérience ?

In [ ]:
from pathlib import Path
import ast
import json

ROOT = Path.cwd()
print(ROOT)
assert (ROOT / "poolbased_surrogate").exists(), "Lance ce notebook depuis la racine du dépôt."

## 1. Vue filesystem

On commence par une carte courte. Les répertoires générés (`.git`, `.venv`, caches) sont ignorés.

In [ ]:
SKIP_DIRS = {".git", ".venv", "__pycache__", ".pytest_cache", "poolbased_surrogate_1d.egg-info", ".claude"}
IMPORTANT_SUFFIXES = {".py", ".yaml", ".yml", ".md", ".ipynb", ".tex", ".bib", ".sh", ".toml"}

files = []
for path in ROOT.rglob("*"):
    if path.is_dir():
        continue
    if any(part in SKIP_DIRS for part in path.parts):
        continue
    if path.suffix in IMPORTANT_SUFFIXES:
        files.append(path.relative_to(ROOT))

by_dir = {}
for path in files:
    top = path.parts[0]
    by_dir.setdefault(top, []).append(path)

for top, items in sorted(by_dir.items()):
    print(f"{top:24s} {len(items):3d} fichiers")
    for item in sorted(items)[:8]:
        print(f"  - {item}")
    if len(items) > 8:
        print(f"  ... +{len(items) - 8}")

## 2. La boucle d'expérience

Le point d'entrée principal est `poolbased_surrogate/run.py`. La boucle fait, à chaque round :

1. créer ou charger la validation ;
2. construire le pool uniforme ou mixte ;
3. calculer les pertes pré-entraînement ;
4. entraîner le surrogate ;
5. entraîner le générateur ;
6. évaluer validation / rollout / hard sets ;
7. sauvegarder `history.json`, `checkpoint_latest.pt` et `pool_round_*.npz`.

In [ ]:
run_path = ROOT / "poolbased_surrogate" / "run.py"
text = run_path.read_text()
for needle in [
    "round 0: using initial uniform pool",
    "generating mixed pool",
    "computing pretrain losses",
    "training surrogate",
    "training ddpm",
    "full validation",
    "np.savez_compressed",
    "save_checkpoint",
]:
    line = next(i for i, line in enumerate(text.splitlines(), start=1) if needle in line)
    print(f"{needle:34s} -> {run_path}:{line}")

## 3. Inventaire des classes et fonctions publiques

Cette cellule extrait les symboles déclarés dans les modules principaux. C'est utile pour lire le code dans le bon ordre.

In [ ]:
MODULES = [
    "poolbased_surrogate/config.py",
    "poolbased_surrogate/pde.py",
    "poolbased_surrogate/data.py",
    "poolbased_surrogate/pool.py",
    "poolbased_surrogate/train.py",
    "poolbased_surrogate/eval.py",
    "poolbased_surrogate/models/surrogate.py",
    "poolbased_surrogate/models/ddpm.py",
    "poolbased_surrogate/metrics.py",
    "poolbased_surrogate/generator_metrics.py",
]

for rel in MODULES:
    path = ROOT / rel
    tree = ast.parse(path.read_text())
    symbols = []
    for node in tree.body:
        if isinstance(node, ast.ClassDef):
            symbols.append(f"class {node.name}")
        elif isinstance(node, ast.FunctionDef):
            symbols.append(f"def {node.name}()")
    print(f"\n{rel}")
    for symbol in symbols[:24]:
        print(f"  {symbol}")
    if len(symbols) > 24:
        print(f"  ... +{len(symbols) - 24}")

## 4. Configs : le langage des expériences

Les configs YAML pilotent tout : PDE, pool, surrogate, générateur, validation, WandB.

Points à surveiller :

- `pool.uniform_fraction` : fraction uniforme du pool ;
- `pool.strategy` : `generator`, `random_tube`, `tube_select`, `mined_ic` ;
- `ddpm.generator` : `ddpm` ou `flow_matching` ;
- `ddpm.mode` : `conditional_loss` ou `conditional_quantile` ;
- `ddpm.difficulty_signal` : `loss` ou `ensemble_var` ;
- `validation.hard_dir` : validations dures optionnelles.

In [ ]:
import yaml

config_paths = sorted((ROOT / "configs").glob("*.yaml"))
print(f"{len(config_paths)} configs trouvées")

interesting = [
    "smoke.yaml",
    "smoke_v3.yaml",
    "bigfoot_ks_phase2_base.yaml",
    "bigfoot_ks_v3_base.yaml",
    "leonardo_burgers_base.yaml",
]
for name in interesting:
    path = ROOT / "configs" / name
    if not path.exists():
        continue
    cfg = yaml.safe_load(path.read_text()) or {}
    print(f"\n{name}")
    print("  output_dir:", cfg.get("output_dir"))
    print("  pde:", cfg.get("pde", {}).get("name"), "res", cfg.get("pde", {}).get("resolution"))
    print("  pool:", cfg.get("pool", {}))
    ddpm = cfg.get("ddpm", {})
    print("  ddpm:", {k: ddpm.get(k) for k in ["enabled", "generator", "mode", "difficulty_signal", "sample_strategy", "sample_mode"] if k in ddpm})

## 5. Artefacts d'un run

Un run complet écrit typiquement :

- `config.resolved.json` : config effectivement utilisée ;
- `history.json` : métriques par round ;
- `checkpoint_latest.pt` : état pour resume ;
- `surrogate.pt`, `ddpm.pt` : poids finaux ;
- `pool_round_<round>.npz` : pool utilisé au round.

In [ ]:
run_dir = ROOT / "runs" / "smoke"
if run_dir.exists():
    for path in sorted(run_dir.iterdir()):
        print(f"{path.name:24s} {path.stat().st_size / 1024:8.1f} KiB")
else:
    print("Pas de runs/smoke. Lance : .venv/bin/python -m poolbased_surrogate.run configs/smoke.yaml --fresh")

## 6. Lecture minimum avant modification

Ordre pragmatique :

1. `HANDOFF.md` pour l'état de recherche ;
2. `docs/roadmap.md` pour la stratégie ;
3. la config de l'expérience que tu veux modifier ;
4. `run.py`, puis seulement le module précis (`pool.py`, `train.py`, `eval.py`, etc.).

In [ ]:
for rel in ["HANDOFF.md", "docs/roadmap.md", "docs/wandb_metrics.md"]:
    path = ROOT / rel
    print(f"\n--- {rel} ---")
    if not path.exists():
        print("missing")
        continue
    lines = path.read_text().splitlines()
    for line in lines[:24]:
        print(line[:120])

## Exercice

Choisis une config dans `configs/`, puis réponds :

- Quel solveur/PDE est utilisé ?
- Quelle fraction du pool reste uniforme ?
- Le générateur conditionne-t-il par perte continue ou par quantile ?
- La validation dure est-elle activée ?
- Quel fichier modifierais-tu pour changer la construction des états générés ?